# Libraries

In [1]:
import datasets
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import col, rtrim, pandas_udf, explode, lit
from functools import reduce
import tiktoken
import pandas as pd
import plotly.express as px
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pyspark.sql.types import ArrayType, StringType
from dotenv import load_dotenv
import os
from openai import OpenAI
import re
import json
import plotly.graph_objects as go

load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI()
embedding_model ="text-embedding-3-small"
encoding = tiktoken.encoding_for_model(embedding_model)

/Users/ozyurtf/Documents/projects/aig-interview-furkan-ozyurt/aig/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Spark Session Initialization

In [2]:
spark = SparkSession.builder\
    .master('local[*]')\
    .appName('AIG_RAG')\
    .config('spark.executor.memory', '8g')\
    .config('spark.driver.memory', '8g')\
    .config('spark.sql.execution.arrow.pyspark.enabled', 'true')\
    .config('spark.sql.shuffle.partitions', '8')\
    .config('spark.driver.maxResultSize', '4g')\
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/19 12:10:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Functions

In [3]:
@pandas_udf("int")
def count_tokens(series: pd.Series) -> pd.Series:
    return series.fillna("").apply(lambda x: len(encoding.encode(x)))

@pandas_udf(ArrayType(StringType()))
def split_chunks(text: pd.Series) -> pd.Series:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=2000, 
        chunk_overlap=100
    )
    chunks = text.apply(lambda t: text_splitter.split_text(t) if t else [])
    return chunks

@pandas_udf("boolean")
def contains_html(series: pd.Series) -> pd.Series:
    pattern = r'<[a-zA-Z][^>]*>|&[a-zA-Z]+;|&#[0-9]+;'
    return series.fillna("").apply(lambda x: bool(re.search(pattern, x)))

# Saving Raw Dataset as Parquet

In [4]:
raw_dataset = datasets.load_dataset("eloukas/edgar-corpus", "full")

Using the latest cached version of the dataset since eloukas/edgar-corpus couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'full' at /Users/ozyurtf/.cache/huggingface/datasets/eloukas___edgar-corpus/full/1.0.0/c2f9ada1db31915d6af4cc19f0ad9486cd0bab93c5c26bb32850e5a1f74f2bd7 (last modified on Wed Apr 15 13:36:41 2026).


In [5]:
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['filename', 'cik', 'year', 'section_1', 'section_1A', 'section_1B', 'section_2', 'section_3', 'section_4', 'section_5', 'section_6', 'section_7', 'section_7A', 'section_8', 'section_9', 'section_9A', 'section_9B', 'section_10', 'section_11', 'section_12', 'section_13', 'section_14', 'section_15'],
        num_rows: 176289
    })
    validation: Dataset({
        features: ['filename', 'cik', 'year', 'section_1', 'section_1A', 'section_1B', 'section_2', 'section_3', 'section_4', 'section_5', 'section_6', 'section_7', 'section_7A', 'section_8', 'section_9', 'section_9A', 'section_9B', 'section_10', 'section_11', 'section_12', 'section_13', 'section_14', 'section_15'],
        num_rows: 22050
    })
    test: Dataset({
        features: ['filename', 'cik', 'year', 'section_1', 'section_1A', 'section_1B', 'section_2', 'section_3', 'section_4', 'section_5', 'section_6', 'section_7', 'section_7A', 'section_8', 'section_9', 'section_9A', '

In [ ]:
raw_dataset["train"].to_parquet("./parquet/edgar-train.parquet")
raw_dataset["validation"].to_parquet("./parquet/edgar-validation.parquet")
raw_dataset["test"].to_parquet("./parquet/edgar-test.parquet")

# Data Processing/Cleaning

In [7]:
sdf_train = spark.read.format("parquet").load('./parquet/edgar-train.parquet').withColumn("split", lit("train"))
sdf_validation = spark.read.format("parquet").load('./parquet/edgar-validation.parquet').withColumn("split", lit("validation"))
sdf_test = spark.read.format("parquet").load('./parquet/edgar-test.parquet').withColumn("split", lit("test"))

In [8]:
train_count = sdf_train.filter((col("cik") == 5272)).count()
validation_count = sdf_validation.filter((col("cik") == 5272)).count()
test_count = sdf_test.filter((col("cik") == 5272)).count()

print(f"Train split has {train_count} AIG records")
print(f"Validation split has {validation_count} AIG records")
print(f"Test split has {test_count} AIG records")

Train split has 22 AIG records
Validation split has 4 AIG records
Test split has 2 AIG records


In [9]:
sdf_train.show(2)

+---------------+------+----+--------------------+----------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+--------------------+----------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+-----+
|       filename|   cik|year|           section_1|section_1A|section_1B|           section_2|           section_3|           section_4|           section_5|           section_6|           section_7|section_7A|           section_8|           section_9|section_9A|section_9B|          section_10|          section_11|          section_12|          section_13|          section_14|section_15|split|
+---------------+------+----+--------------------+----------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------

In [10]:
sdf = reduce(DataFrame.unionByName, [sdf_train, sdf_validation, sdf_test])

In [11]:
sdf.filter(col("cik").startswith("0")).count()

0

In [12]:
count_5272 = sdf.filter((col("cik") == 5272)).count()
count_1000617 = sdf.filter((col("cik") == 1000617)).count()
count_1503633 = sdf.filter((col("cik") == 1503633)).count()

print(f"Entire dataset has: \n - {count_5272} records with CIK = 5272 \n - {count_1000617} records with CIK = 1000617 \n - {count_1503633} records with CIK = 1503633")

Entire dataset has: 
 - 28 records with CIK = 5272 
 - 0 records with CIK = 1000617 
 - 0 records with CIK = 1503633


In [13]:
sdf_aig = sdf.filter(col("cik") == 5272).drop("split", "filename").melt(
    ids = ["cik", "year"], 
    values =  ['section_1', 'section_1A', 'section_1B',
               'section_2', 'section_3', 'section_4',
               'section_5', 'section_6', 'section_7',
               'section_7A', 'section_8', 'section_9',
               'section_9A', 'section_9B', 'section_10',
               'section_11', 'section_12', 'section_13',
               'section_14', 'section_15'],
    variableColumnName = "section",
    valueColumnName = "description"
)

In [14]:
sdf_aig.show()

+----+----+----------+--------------------+
| cik|year|   section|         description|
+----+----+----------+--------------------+
|5272|1994| section_1|ITEM 1. BUSINESS\...|
|5272|1994|section_1A|                    |
|5272|1994|section_1B|                    |
|5272|1994| section_2|ITEM 2. PROPERTIE...|
|5272|1994| section_3|ITEM 3. LEGAL PRO...|
|5272|1994| section_4|ITEM 4. SUBMISSIO...|
|5272|1994| section_5|ITEM 5. MARKET FO...|
|5272|1994| section_6|ITEM 6. SELECTED ...|
|5272|1994| section_7|ITEM 7. MANAGEMEN...|
|5272|1994|section_7A|                    |
|5272|1994| section_8|ITEM 8. FINANCIAL...|
|5272|1994| section_9|ITEM 9. CHANGES I...|
|5272|1994|section_9A|                    |
|5272|1994|section_9B|                    |
|5272|1994|section_10|ITEM 10. DIRECTOR...|
|5272|1994|section_11|ITEM 11. EXECUTIV...|
|5272|1994|section_12|ITEM 12. SECURITY...|
|5272|1994|section_13|ITEM 13. CERTAIN ...|
|5272|1994|section_14|ITEM 14. EXHIBITS...|
|5272|1994|section_15|          

In [15]:
sdf_aig \
    .withColumn("contain_html", contains_html(col("description"))) \
    .filter(col("contain_html") == True) \
    .count()

2

In [16]:
sdf_aig = sdf_aig.withColumn("description", rtrim(col("description")))

In [17]:
null_counts = {c: sdf_aig.filter(col(c).isNull()).count() for c in sdf_aig.columns} 
print(null_counts)

{'cik': 0, 'year': 0, 'section': 0, 'description': 0}


In [18]:
empty_counts = sdf_aig.where(col("description") == "").count()
print(empty_counts)

86


In [19]:
sdf_aig = sdf_aig.na.drop(subset=['description']).filter(col('description') != "")

In [20]:
sdf_aig.cache()

DataFrame[cik: string, year: string, section: string, description: string]

In [21]:
sdf_aig.show(10)

+----+----+----------+--------------------+
| cik|year|   section|         description|
+----+----+----------+--------------------+
|5272|1994| section_1|ITEM 1. BUSINESS\...|
|5272|1994| section_2|ITEM 2. PROPERTIE...|
|5272|1994| section_3|ITEM 3. LEGAL PRO...|
|5272|1994| section_4|ITEM 4. SUBMISSIO...|
|5272|1994| section_5|ITEM 5. MARKET FO...|
|5272|1994| section_6|ITEM 6. SELECTED ...|
|5272|1994| section_7|ITEM 7. MANAGEMEN...|
|5272|1994| section_8|ITEM 8. FINANCIAL...|
|5272|1994| section_9|ITEM 9. CHANGES I...|
|5272|1994|section_10|ITEM 10. DIRECTOR...|
+----+----+----------+--------------------+
only showing top 10 rows


In [22]:
sdf_aig.count()

474

In [23]:
sdf_aig.select(
    col("year"),
    col("section"),
    col("description"),
    count_tokens(col("description")).alias("token_count")
).show()

+----+----------+--------------------+-----------+
|year|   section|         description|token_count|
+----+----------+--------------------+-----------+
|1994| section_1|ITEM 1. BUSINESS\...|       6102|
|1994| section_2|ITEM 2. PROPERTIE...|        158|
|1994| section_3|ITEM 3. LEGAL PRO...|        104|
|1994| section_4|ITEM 4. SUBMISSIO...|        399|
|1994| section_5|ITEM 5. MARKET FO...|        407|
|1994| section_6|ITEM 6. SELECTED ...|        222|
|1994| section_7|ITEM 7. MANAGEMEN...|      16964|
|1994| section_8|ITEM 8. FINANCIAL...|      19790|
|1994| section_9|ITEM 9. CHANGES I...|         64|
|1994|section_10|ITEM 10. DIRECTOR...|         89|
|1994|section_11|ITEM 11. EXECUTIV...|         59|
|1994|section_12|ITEM 12. SECURITY...|         67|
|1994|section_13|ITEM 13. CERTAIN ...|         67|
|1994|section_14|ITEM 14. EXHIBITS...|       1093|
|1995| section_1|ITEM 1. BUSINESS\...|       6107|
|1995| section_2|ITEM 2. PROPERTIE...|        158|
|1995| section_3|ITEM 3. LEGAL 

In [24]:
pdf = sdf_aig.select(
    count_tokens(col("description")).alias("token_count")
).toPandas()

In [25]:
fig = px.box(
    pdf,
    x="token_count",  
    points="outliers",
    title="Token Count Distribution (Horizontal Box Plot)"
)

fig.update_layout(
    xaxis_title="Token Count",
    title_x=0.5,
    template="plotly_white"
)

fig.show()

In [26]:
q = 70
avg_chunk_size = 500

q_token_count = int(pdf["token_count"].quantile(q / 100))
total_tokens = pdf["token_count"].sum()
num_chunks = total_tokens // avg_chunk_size

print(f"{q}% of the sections has less than or equal to ~{q_token_count} number of tokens.")
print(f"Total number of tokens across all the sections is: {total_tokens}.")
print(f"This means that if we want the average chunk size to be ~{avg_chunk_size}, for instance, we will have ~{num_chunks} number of chunks.")

70% of the sections has less than or equal to ~966 number of tokens.
Total number of tokens across all the sections is: 3491821.
This means that if we want the average chunk size to be ~500, for instance, we will have ~6983 number of chunks.


In [27]:
sdf_aig_final = sdf_aig \
    .withColumn("chunks", split_chunks(col("description"))) \
    .withColumn("chunks", explode(col("chunks"))) \
    .drop("cik", "description")

In [28]:
sdf_aig_final.show()

+----+---------+--------------------+
|year|  section|              chunks|
+----+---------+--------------------+
|1994|section_1|ITEM 1. BUSINESS\...|
|1994|section_1|At December 31, 1...|
|1994|section_1|AIG's business de...|
|1994|section_1|American Internat...|
|1994|section_1|AIG's general ins...|
|1994|section_1|The following tab...|
|1994|section_1|Loss reserves est...|
|1994|section_1|had actually been...|
|1994|section_1|Approximately 50 ...|
|1994|section_1|LIFE INSURANCE OP...|
|1994|section_1|Traditional life ...|
|1994|section_1|(c) Includes $61,...|
|1994|section_1|AGENCY AND SERVIC...|
|1994|section_1|FINANCIAL SERVICE...|
|1994|section_1|* Represents net ...|
|1994|section_1|LOCATIONS OF CERT...|
|1994|section_1|AIG's insurance s...|
|1994|section_1|The RBC Model Law...|
|1994|section_1|In addition to li...|
|1994|section_2|ITEM 2. PROPERTIE...|
+----+---------+--------------------+
only showing top 20 rows


In [29]:
sdf_aig_final.count()

11162

In [30]:
sdf_aig_final.select(
    col("year"),
    col("section"),
    col("chunks"),
    count_tokens(col("chunks")).alias("token_count")
).show()

+----+---------+--------------------+-----------+
|year|  section|              chunks|token_count|
+----+---------+--------------------+-----------+
|1994|section_1|ITEM 1. BUSINESS\...|        329|
|1994|section_1|At December 31, 1...|        368|
|1994|section_1|AIG's business de...|        306|
|1994|section_1|American Internat...|        321|
|1994|section_1|AIG's general ins...|        325|
|1994|section_1|The following tab...|        336|
|1994|section_1|Loss reserves est...|        273|
|1994|section_1|had actually been...|        371|
|1994|section_1|Approximately 50 ...|        356|
|1994|section_1|LIFE INSURANCE OP...|        349|
|1994|section_1|Traditional life ...|        383|
|1994|section_1|(c) Includes $61,...|        314|
|1994|section_1|AGENCY AND SERVIC...|        217|
|1994|section_1|FINANCIAL SERVICE...|        345|
|1994|section_1|* Represents net ...|        377|
|1994|section_1|LOCATIONS OF CERT...|        320|
|1994|section_1|AIG's insurance s...|        330|


In [31]:
pdf_final = sdf_aig_final.select(
    count_tokens(col("chunks")).alias("token_count")
).toPandas()

In [32]:
fig = px.box(
    pdf_final,
    x="token_count",  
    points="outliers",
    title="Token Count Distribution (Horizontal Box Plot)"
)

fig.update_layout(
    xaxis_title="Token Count",
    title_x=0.5,
    template="plotly_white"
)

fig.show()

In [33]:
max_token_count = max(pdf_final["token_count"])
min_token_count = min(pdf_final["token_count"])
avg_token_count = np.round(np.mean(pdf_final["token_count"]), 2)
median_token_count = np.median(pdf_final["token_count"])

print(f"After the chunking process: \n - The minimum number of tokens: {min_token_count} \n - The maximum number of tokens: {max_token_count} \n - Average number of tokens: {avg_token_count} \n - Median token number: {median_token_count}")

After the chunking process: 
 - The minimum number of tokens: 5 
 - The maximum number of tokens: 792 
 - Average number of tokens: 317.3 
 - Median token number: 333.0


# Ground Truth Data

### Numerical Variable - 1

In [34]:
description = sdf_aig.filter(
    (col("year") == 2012) & (col("section") == "section_2")
).select("description").collect()[0]["description"]

print(description)

ITEM 2 / PROPERTIES
AIG and its subsidiaries operate from over 400 offices in the United States and approximately 600 offices in over 75 foreign countries. The following offices are located in buildings in the United States owned by AIG and its subsidiaries:
AIG Property Casualty:
AIG Life and Retirement:
•
175 Water Street in New York, New York
•
Amarillo, Ft. Worth and Houston, Texas
•
Wilmington, Delaware
•
Nashville, Tennessee
•
Stevens Point, Wisconsin
•
San Juan, Puerto Rico
Other Operations:
•
Greensboro and Winston-Salem, North Carolina
•
Livingston, New Jersey
•
Stowe, Vermont
In addition, AIG Property Casualty owns offices in approximately 20 foreign countries and jurisdictions including Argentina, Bermuda, Colombia, Ecuador, Japan, Mexico, the U.K., Taiwan, and Venezuela. The remainder of the office space utilized by AIG and its subsidiaries is leased. AIG believes that its leases and properties are sufficient for its current purposes.
LOCATIONS OF CERTAIN ASSETS
As of Decem

In [35]:
description = sdf_aig.filter(
    (col("year") == 2013) & (col("section") == "section_2")
).select("description").collect()[0]["description"]

print(description)

ITEM 2 / PROPERTIES
AIG and its subsidiaries operate from over 400 offices in the United States and approximately 600 offices in over 75 foreign countries. The following offices are located in buildings in the United States owned by AIG and its subsidiaries:
AIG Property Casualty:
AIG Life and Retirement:
•
175 Water Street in New York, New York
•
Amarillo, Ft. Worth and Houston, Texas
•
Wilmington, Delaware
•
Nashville, Tennessee
•
Stevens Point, Wisconsin
•
San Juan, Puerto Rico
Other Operations:
•
Greensboro and Winston-Salem, North Carolina
•
Livingston, New Jersey
•
Stowe, Vermont
In addition, AIG Property Casualty owns offices in approximately 20 foreign countries and jurisdictions including Argentina, Bermuda, Colombia, Ecuador, Japan, Mexico, the U.K., Taiwan, and Venezuela. The remainder of the office space utilized by AIG and its subsidiaries is leased. AIG believes that its leases and properties are sufficient for its current purposes.
LOCATIONS OF CERTAIN ASSETS
As of Decem

In [36]:
description = sdf_aig.filter(
    (col("year") == 2014) & (col("section") == "section_2")
).select("description").collect()[0]["description"]

print(description)

ITEM 2 / PROPERTIES
We operate from approximately 400 offices in the United States and approximately 500 offices in over 75 foreign countries. The following offices are located in buildings in the United States owned by us:
Non-Life Insurance Companies:
• Wilmington, Delaware
• Stevens Point, Wisconsin
• Greensboro and Winston-Salem, North Carolina
Life Insurance Companies:
• Amarillo and Houston, Texas
Corporate and Other:
• 175 Water Street in New York, New York
• Livingston, New Jersey
• Stowe, Vermont
• Ft. Worth, Texas
In addition, Non-Life Insurance Companies own offices in approximately 20 foreign countries and jurisdictions including Argentina, Bermuda, Colombia, Ecuador, Japan, Mexico, the U.K., Taiwan, and Venezuela. The remainder of the office space we utilize is leased. We believe that our leases and properties are sufficient for our current purposes.
LOCATIONS OF CERTAIN ASSETS
As of December 31, 2014, approximately 10 percent of our consolidated assets were located outsid

In [37]:
description = sdf_aig.filter(
    (col("year") == 2016) & (col("section") == "section_2")
).select("description").collect()[0]["description"]

print(description)

ITEM 2 | Properties
We operate from approximately 180 offices in the United States and approximately 500 offices in over 70 foreign countries. The following offices are located in buildings in the United States owned by us:
Property Casualty Insurance Companies:
• Stevens Point, Wisconsin
Life Insurance Companies:
• Amarillo and Houston, Texas
Other Operations:
• 175 Water Street in New York, New York
• Livingston, New Jersey
• Stowe, Vermont
• Ft. Worth, Texas
In addition, our Property Casualty Insurance Companies own offices in approximately 20 foreign countries and jurisdictions including Argentina, Bermuda, Colombia, Ecuador, Japan, Mexico, the UK and Venezuela. The remainder of the office space we use is leased. We believe that our leases and properties are sufficient for our current purposes.
LOCATIONS OF CERTAIN ASSETS
As of December 31, 2016, approximately 11 percent of our consolidated assets were located outside the U.S. and Canada, including $532 million of cash and securiti

In [38]:
description = sdf_aig.filter(
    (col("year") == 2017) & (col("section") == "section_2")
).select("description").collect()[0]["description"]

print(description)

ITEM 2 | Properties
We operate from approximately 160 offices in the United States and approximately 380 offices in approximately 60 foreign countries. The following offices are located in buildings in the United States owned by us:
General Insurance Companies:
• Stevens Point, Wisconsin
Life and Retirement Companies:
• Amarillo and Houston, Texas
Other Operations:
• 175 Water Street in New York, New York (Corporate Headquarters; also includes General Insurance companies)
• Livingston, New Jersey
• Ft. Worth, Texas
In addition, our General Insurance companies own offices in 13 foreign countries and jurisdictions including Bermuda, Ecuador, Japan, Mexico, the UK and Venezuela. The remainder of the office space we use is leased. We believe that our leases and properties are sufficient for our current purposes.
LOCATIONS OF CERTAIN ASSETS
As of December 31, 2017, approximately 11 percent of our consolidated assets were located outside the U.S. and Canada, including $491 million of cash an

### Numerical Variable - 2

In [39]:
description = sdf_aig.filter(
    (col("year") == 2012) & (col("section") == "section_7")
).select("description").collect()[0]["description"]

print(description)

Item 7. MD&A - Results of Operations - AIG Property Casualty Operations for a reconciliation of the adjusted combined ratio.
AIG 2012 Form 10-K
Changes In Accounting For Acquisition Costs
Reflects changes from the adoption of the new accounting standard related to deferred acquisition costs for 2009 and 2008, as set out in further detail below. See Note 2 to the Consolidated Financial Statements for a description of the effect of the adoption of the new accounting standards on 2011 and 2010 periods, which is also reflected in the data presented above.
(a) Includes the effect of the reclassification of ILFC as discontinued operations.
(b) Includes an adjustment to the loss accrual related to the sale of Nan Shan of $2.3 billion.
AIG 2012 Form 10-K
Items Affecting Comparability Between Periods
The following are significant developments that affected multiple periods and financial statement captions. Other items that affected comparability are included in the footnotes to the table presen

In [40]:
description = sdf_aig.filter(
    (col("year") == 2015) & (col("section") == "section_7")
).select("description").collect()[0]["description"]

print(description)

Item 7 / LIQUIDITY AND CAPITAL rESOURCES
Dividends and Repurchases of AIG Common Stock
On February 12, 2015, our Board of Directors declared a cash dividend on AIG Common Stock of $0.125 per share, payable on March 26, 2015 to shareholders of record on March 12, 2015. On April 30, 2015, our Board of Directors declared a cash dividend on AIG Common Stock of $0.125 per share, payable on June 25, 2015 to shareholders of record on June 11, 2015. On August 3, 2015, our Board of Directors declared a cash dividend on AIG Common Stock of $0.28 per share, payable on September 28, 2015 to shareholders of record on September 14, 2015. On November 2, 2015, our Board of Directors declared a cash dividend on AIG Common Stock of $0.28 per share, payable on December 21, 2015 to shareholders of record on December 7, 2015.
On February 11, 2016, our Board of Directors declared a cash dividend on AIG Common Stock of $0.32 per share, payable on March 28, 2016 to shareholders of record on March 14, 2016. Th

In [41]:
description = sdf_aig.filter(
    (col("year") == 2016) & (col("section") == "section_7")
).select("description").collect()[0]["description"]

print(description)

ITEM 7 | Use of Non-GAAP Measures
comparisons with our insurance competitors. When we use these measures, reconciliations to the most comparable GAAP measure are provided on a consolidated basis in the Results of Operations section of this MD&A.
Operating revenues exclude Net realized capital gains (losses), income from non-operating litigation settlements (included in Other income for GAAP purposes) and changes in fair value of securities used to hedge guaranteed living benefits (included in Net investment income for GAAP purposes). Operating revenues are a GAAP measure for our operating segments.
Pre-tax operating income is derived by excluding the following items from income from continuing operations before income tax. This definition is consistent across our modules (including geography). These items generally fall into one or more of the following broad categories: legacy matters having no relevance to our current businesses or operating performance; adjustments to enhance transp

In [42]:
description = sdf_aig.filter(
    (col("year") == 2017) & (col("section") == "section_7")
).select("description").collect()[0]["description"]

print(description)

ITEM 7 | Use of Non-GAAP Measures
We use the following operating performance measures because we believe they enhance the understanding of the underlying profitability of continuing operations and trends of our business segments. We believe they also allow for more meaningful comparisons with our insurance competitors. When we use these measures, reconciliations to the most comparable GAAP measure are provided on a consolidated basis in the Consolidated Results of Operations section of this MD&A.
Adjusted revenues exclude Net realized capital gains (losses), income from non-operating litigation settlements (included in Other income for GAAP purposes) and changes in fair value of securities used to hedge guaranteed living benefits (included in Net investment income for GAAP purposes). Adjusted revenues is a GAAP measure for our operating segments.
Adjusted pre-tax income is derived by excluding the following items from income from continuing operations before income tax. This definition

In [43]:
description = sdf_aig.filter(
    (col("year") == 2018) & (col("section") == "section_7")
).select("description").collect()[0]["description"]

print(description)

ITEM 7 | Management’s Discussion and Analysis of Financial Condition and Results of Operations
Cautionary Statement Regarding Forward-Looking Information
This Annual Report on Form 10-K and other publicly available documents may include, and officers and representatives of AIG may from time to time make and discuss, projections, goals, assumptions and statements that may constitute “forward-looking statements” within the meaning of the Private Securities Litigation Reform Act of 1995. These projections, goals, assumptions and statements are not historical facts but instead represent only a belief regarding future events, many of which, by their nature, are inherently uncertain and outside our control. These projections, goals, assumptions and statements include statements preceded by, followed by or including words such as “will,” “believe,” “anticipate,” “expect,” “intend,” “plan,” “focused on achieving,” “view,” “target,” “goal” or “estimate.” These projections, goals, assumptions an

### Categorical Variable - 1

In [44]:
description = sdf_aig.filter(
    (col("year") == 1995) & (col("section") == "section_2")
).select("description").collect()[0]["description"]

print(description)

ITEM 2. PROPERTIES
AIG and its subsidiaries operate from approximately 250 offices in the United States, 5 offices in Canada and numerous offices in other foreign countries. The offices in Manchester, New Hampshire; Springfield, Illinois; Houston, Texas; Atlanta, Georgia; Baton Rouge, Louisiana; Wilmington, Delaware; Hato Rey, Puerto Rico; San Diego, California; Greensboro, North Carolina; Livingston, New Jersey; 70 Pine Street and 72 Wall Street in New York City; and offices in approximately 30 foreign countries including Bermuda, Hong Kong, the Philippines, Japan, England, Singapore, Taiwan and Thailand are located in buildings owned by AIG and its subsidiaries. The remainder of the office space utilized by AIG subsidiaries is leased.
ITEM 3.


In [45]:
description = sdf_aig.filter(
    (col("year") == 1999) & (col("section") == "section_2")
).select("description").collect()[0]["description"]

print(description)

ITEM 2. Properties
AIG and its subsidiaries operate from approximately 360 offices in the United States, 5 offices in Canada and numerous offices in approximately 100 foreign countries. The offices in Springfield, Illinois; Houston, Texas; Atlanta, Georgia; Baton Rouge, Louisiana; Wilmington, Delaware; Hato Rey, Puerto Rico; San Diego, California; Greensboro, North Carolina; Livingston, New Jersey; 70 Pine Street, 72 Wall Street and 175 Water Street in New York City; and offices in approximately 30 foreign countries including Bermuda, Chile, Hong Kong, the Philippines, Japan, England, Singapore, Switzerland, Taiwan and Thailand are located in buildings owned by AIG and its subsidiaries. The remainder of the office space utilized by AIG subsidiaries is leased.
ITEM 3.


In [46]:
description = sdf_aig.filter(
    (col("year") == 2004) & (col("section") == "section_2")
).select("description").collect()[0]["description"]

print(description)

ITEM 2.
Properties
AIG and its subsidiaries operate from approximately 2,200 offices in the United States, 8 offices in Canada and numerous offices in approximately 100 foreign countries. The offices in Springfield, Illinois; Amarillo, Ft. Worth and Houston, Texas; Wilmington, Delaware; Hato Rey and San Juan, Puerto Rico; Tampa, Florida; Livingston, New Jersey; Evansville, Indiana; Nashville, Tennessee; 70 Pine Street, 72 Wall Street and 175 Water Street in New York City; and offices in more than 30 foreign countries including Bermuda, Chile, Hong Kong, the Philippines, Japan, United Kingdom, Singapore, Switzerland, Taiwan and Thailand are located in buildings owned by AIG and its subsidiaries. The remainder of the office space utilized by AIG subsidiaries is leased.
ITEM 3.


In [47]:
description = sdf_aig.filter(
    (col("year") == 2010) & (col("section") == "section_2")
).select("description").collect()[0]["description"]

print(description)

Item 2. Properties
AIG and its subsidiaries operate from approximately 500 offices in the United States and numerous offices in over 75 foreign countries. The following offices are located in buildings owned by AIG and its subsidiaries:
Greensboro and Winston-Salem, North Carolina
Nashville, Tennessee
Amarillo, Ft. Worth and Houston, Texas
Stevens Point, Wisconsin
San Juan, Puerto Rico
175 Water Street in New York, New York
Livingston, New Jersey
In addition, offices in more than 30 foreign countries and jurisdictions including Bermuda, Chile, Hong Kong, the Philippines, Japan, the U.K., Singapore, Malaysia, Taiwan and Thailand are located in buildings owned by AIG and its subsidiaries. The remainder of the office space utilized by AIG and its subsidiaries is leased. AIG believes that its leases and properties are sufficient for its current purposes.
Item 3.


In [48]:
description = sdf_aig.filter(
    (col("year") == 2016) & (col("section") == "section_2")
).select("description").collect()[0]["description"]

print(description)

ITEM 2 | Properties
We operate from approximately 180 offices in the United States and approximately 500 offices in over 70 foreign countries. The following offices are located in buildings in the United States owned by us:
Property Casualty Insurance Companies:
• Stevens Point, Wisconsin
Life Insurance Companies:
• Amarillo and Houston, Texas
Other Operations:
• 175 Water Street in New York, New York
• Livingston, New Jersey
• Stowe, Vermont
• Ft. Worth, Texas
In addition, our Property Casualty Insurance Companies own offices in approximately 20 foreign countries and jurisdictions including Argentina, Bermuda, Colombia, Ecuador, Japan, Mexico, the UK and Venezuela. The remainder of the office space we use is leased. We believe that our leases and properties are sufficient for our current purposes.
LOCATIONS OF CERTAIN ASSETS
As of December 31, 2016, approximately 11 percent of our consolidated assets were located outside the U.S. and Canada, including $532 million of cash and securiti

# Performance Comparison with Visualization

In [49]:
path1 = "eval/config_alpha-0.0_topk-10_temperature-0.0"
path2 = "eval/config_alpha-0.5_topk-10_temperature-0.0"
path3 = "eval/config_alpha-1.0_topk-10_temperature-0.0"

with open(path1 + "/scores.json", "r") as f:
    scores1 = json.load(f)

with open(path2 + "/scores.json", "r") as f:
    scores2 = json.load(f)    

with open(path3 + "/scores.json", "r") as f:
    scores3 = json.load(f)        

In [50]:
metric_labels = {
    "recall_k": "Recall@k",
    "mrr": "MRR",
    "exact_match_score": "Exact Match"
}

metrics = list(metric_labels.keys())
pretty_names = [metric_labels[m] for m in metrics]

def extract_scores(scores):
    return {
        "recall_k": scores["retriever"]["recall_k"],
        "mrr": scores["retriever"]["mrr"],
        "exact_match_score": scores["generator"]["exact_match_score"],
    }

s1 = extract_scores(scores1)
s2 = extract_scores(scores2)
s3 = extract_scores(scores3)

fig = go.Figure()

fig.add_trace(go.Bar(
    name="BM25 (100%)",
    x=pretty_names,
    y=[s1[m] for m in metrics],
))

fig.add_trace(go.Bar(
    name="BM25 (50%) + Dense Embedding (50%)",
    x=pretty_names,
    y=[s2[m] for m in metrics],
))

fig.add_trace(go.Bar(
    name="Dense Embedding (100%)",
    x=pretty_names,
    y=[s3[m] for m in metrics],
))

fig.update_layout(
    title="RAG Evaluation Comparison",
    title_x=0.5,
    barmode="group",
    xaxis_title="Metrics",
    yaxis_title="Score",
    template="plotly_white",
    width=1300,
    height=500,
    margin=dict(l=40, r=40, t=60, b=40),
    legend_title="Runs"
)

fig.show()